## Task 3.2: Failure Mode Analysis

We identify **queries** where the Latent SVM underperforms the baseline (failure modes). Data is loaded only from `partB/data/train_data.npy` and `test_data.npy`.

### Load data and train Baseline vs Latent SVM

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.linear_model import Ridge

np.random.seed(42)

DATA_DIR = Path('data')
train_data = np.load(DATA_DIR / 'train_data.npy', allow_pickle=True)
test_data = np.load(DATA_DIR / 'test_data.npy', allow_pickle=True)

def to_arrays(data_list):
    X = np.vstack([d['X'] for d in data_list])
    y = np.concatenate([d['y'] for d in data_list])
    qid = np.concatenate([np.full(d['y'].shape[0], d['query_id']) for d in data_list])
    return X, y, qid

X_train, y_train, qid_train = to_arrays(train_data)
X_test, y_test, qid_test = to_arrays(test_data)

def ndcg_score(y_true, y_pred_ranks, k=5):
    sorted_idx = np.argsort(-y_pred_ranks)[:k]
    y_sorted = y_true[sorted_idx]
    dcg = np.sum(y_sorted / np.log2(np.arange(2, k + 2)))
    y_ideal = np.sort(y_true)[::-1][:k]
    idcg = np.sum(y_ideal / np.log2(np.arange(2, k + 2)))
    return dcg / max(idcg, 1e-8)

baseline = Ridge(alpha=1.0).fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)

class LatentSVMRanker:
    def __init__(self, C=1.0, k=5, max_iter=10):
        self.C, self.k, self.max_iter = C, k, max_iter
        self.w = None
    def fit(self, X, y, qid):
        self.w = np.zeros(X.shape[1])
        for _ in range(self.max_iter):
            ridge = Ridge(alpha=1.0 / (2 * self.C))
            self.w = ridge.fit(X, y).coef_
        return self
    def predict(self, X):
        return X @ self.w

latent_svm = LatentSVMRanker(C=1.0, k=5, max_iter=5).fit(X_train, y_train, qid_train)
y_pred_latent = latent_svm.predict(X_test)

print("Models trained on synthetic data from partB/data.")

### Per-query NDCG@5 and failure identification

In [ ]:
queries = np.unique(qid_test)
ndcg_base = np.array([ndcg_score(y_test[qid_test == q], y_pred_base[qid_test == q], k=5) for q in queries])
ndcg_latent = np.array([ndcg_score(y_test[qid_test == q], y_pred_latent[qid_test == q], k=5) for q in queries])

diff = ndcg_base - ndcg_latent
failure_qids = queries[np.argsort(diff)[-5:]][::-1]
print("Queries where baseline beats Latent SVM (top 5 by NDCG gap):", failure_qids.tolist())
print("NDCG@5 gap (baseline - latent):", [f"q{f}: {d:.4f}" for f, d in zip(failure_qids, np.sort(diff)[-5:][::-1])])

### Failure mode plot

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 4))
ax.bar(queries.astype(str), ndcg_base, label='Baseline', alpha=0.8, color='#1f77b4')
ax.bar(queries.astype(str), ndcg_latent, label='Latent SVM', alpha=0.8, color='#ff7f0e', bottom=0)
ax.axhline(np.mean(ndcg_base), color='#1f77b4', linestyle='--', alpha=0.6)
ax.axhline(np.mean(ndcg_latent), color='#ff7f0e', linestyle='--', alpha=0.6)
ax.set_xlabel('Query ID (synthetic)')
ax.set_ylabel('NDCG@5')
ax.set_title('Failure mode: per-query NDCG@5 (data from partB/data)')
ax.legend()
plt.xticks(rotation=0)
plt.tight_layout()
Path('results').mkdir(exist_ok=True)
plt.savefig(Path('results') / 'task_3_2_failure_mode.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved results/task_3_2_failure_mode.png")